In [1]:
%%capture
!pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
!pip install --no-deps unsloth

In [2]:
import os
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B",
    max_seq_length = 2048,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


    PyTorch 2.6.0+cu124 with CUDA 1204 (you have 2.8.0+cu126)
    Python  3.12.9 (you have 3.12.12)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


Switching to PyTorch attention since your Xformers is broken.

Unsloth: Xformers was not installed correctly.
Please install xformers separately first.
Then confirm if it's correctly installed by running:
python -m xformers.info

Longer error message:
xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.6.0+cu124 with CUDA 1204 (you have 2.8.0+cu126)
    Python  3.12.9 (you have 3.12.12)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.10.7: Fast Qwen3 patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/

model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2025.10.7 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [4]:
from datasets import load_dataset
reasoning_dataset = load_dataset("unsloth/OpenMathReasoning-mini", split = "cot")
non_reasoning_dataset = load_dataset("mlabonne/FineTome-100k", split = "train")

In [5]:
from pprint import pprint
pprint(reasoning_dataset[0])

{'expected_answer': '14',
 'generated_solution': '<think>\n'
                       "Okay, let's see. I need to solve the equation √(x² + "
                       '165) - √(x² - 52) = 7, and find all positive values of '
                       'x. Hmm, radicals can be tricky, but maybe if I can '
                       'eliminate the square roots by squaring both sides. Let '
                       'me try that.\n'
                       '\n'
                       'First, let me write down the equation again to make '
                       'sure I have it right:\n'
                       '\n'
                       '√(x² + 165) - √(x² - 52) = 7.\n'
                       '\n'
                       'Okay, so the idea is to isolate one of the radicals '
                       'and then square both sides. Let me try moving the '
                       'second radical to the other side:\n'
                       '\n'
                       '√(x² + 165) = 7 + √(x² - 52).\n'
             

In [6]:
pprint(non_reasoning_dataset[0])

{'conversations': [{'from': 'human',
                    'value': 'Explain what boolean operators are, what they '
                             'do, and provide examples of how they can be used '
                             'in programming. Additionally, describe the '
                             'concept of operator precedence and provide '
                             'examples of how it affects the evaluation of '
                             'boolean expressions. Discuss the difference '
                             'between short-circuit evaluation and normal '
                             'evaluation in boolean expressions and '
                             'demonstrate their usage in code. \n'
                             '\n'
                             'Furthermore, add the requirement that the code '
                             'must be written in a language that does not '
                             'support short-circuit evaluation natively, '
                        

In [7]:
def generate_conversation(examples):
    problems  = examples["problem"]
    solutions = examples["generated_solution"]
    conversations = []
    for problem, solution in zip(problems, solutions):
        conversations.append([
            {"role" : "user",      "content" : problem},
            {"role" : "assistant", "content" : solution},
        ])
    return { "conversations": conversations, }

In [8]:
reasoning_conversations = [tokenizer.apply_chat_template(
    conversation,
    tokenize = False,
) for conversation in reasoning_dataset.map(generate_conversation, batched = True)["conversations"]]

In [9]:
print(reasoning_conversations[0])

<|im_start|>user
Given $\sqrt{x^2+165}-\sqrt{x^2-52}=7$ and $x$ is positive, find all possible values of $x$.<|im_end|>
<|im_start|>assistant
<think>
Okay, let's see. I need to solve the equation √(x² + 165) - √(x² - 52) = 7, and find all positive values of x. Hmm, radicals can be tricky, but maybe if I can eliminate the square roots by squaring both sides. Let me try that.

First, let me write down the equation again to make sure I have it right:

√(x² + 165) - √(x² - 52) = 7.

Okay, so the idea is to isolate one of the radicals and then square both sides. Let me try moving the second radical to the other side:

√(x² + 165) = 7 + √(x² - 52).

Now, if I square both sides, maybe I can get rid of the square roots. Let's do that:

(√(x² + 165))² = (7 + √(x² - 52))².

Simplifying the left side:

x² + 165 = 49 + 14√(x² - 52) + (√(x² - 52))².

The right side is expanded using the formula (a + b)² = a² + 2ab + b². So the right side becomes 7² + 2*7*√(x² - 52) + (√(x² - 52))², which is 49 + 14

In [10]:
from unsloth.chat_templates import standardize_sharegpt
dataset = standardize_sharegpt(non_reasoning_dataset)

non_reasoning_conversations = [tokenizer.apply_chat_template(
    conversation,
    tokenize = False,
) for conversation in dataset["conversations"]]

In [11]:
print(non_reasoning_conversations[0])

<|im_start|>user
Explain what boolean operators are, what they do, and provide examples of how they can be used in programming. Additionally, describe the concept of operator precedence and provide examples of how it affects the evaluation of boolean expressions. Discuss the difference between short-circuit evaluation and normal evaluation in boolean expressions and demonstrate their usage in code. 

Furthermore, add the requirement that the code must be written in a language that does not support short-circuit evaluation natively, forcing the test taker to implement their own logic for short-circuit evaluation.

Finally, delve into the concept of truthiness and falsiness in programming languages, explaining how it affects the evaluation of boolean expressions. Add the constraint that the test taker must write code that handles cases where truthiness and falsiness are implemented differently across different programming languages.<|im_end|>
<|im_start|>assistant
<think>

</think>

Bool

In [12]:
print(len(reasoning_conversations))
print(len(non_reasoning_conversations))

19252
100000


In [13]:
chat_percentage = 0.75

In [14]:
import pandas as pd
non_reasoning_subset = pd.Series(non_reasoning_conversations)
non_reasoning_subset = non_reasoning_subset.sample(
    int(len(reasoning_conversations) * (1.0 - chat_percentage)),
    random_state = 2407,
)

In [15]:
data = pd.concat([
    pd.Series(reasoning_conversations),
    pd.Series(non_reasoning_subset)
])
data.name = "text"

from datasets import Dataset
combined_dataset = Dataset.from_pandas(pd.DataFrame(data))
combined_dataset = combined_dataset.shuffle(seed = 3407)

In [16]:
len(combined_dataset)

24065

In [17]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = combined_dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 30,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    )
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/24065 [00:00<?, ? examples/s]

In [18]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 24,065 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,0.615100
2,0.678900
3,0.857600
4,0.697800
5,0.581400
6,0.549100
7,0.549600
8,0.507500
9,0.468300
10,0.575600


In [19]:
messages = [
    {"role" : "user", "content" : "Solve (x^2 + 5x + 6 = 0)."}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,
    enable_thinking = False,
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 2048,
    temperature = 0.7, top_p = 0.8, top_k = 20,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

To solve the quadratic equation \( x^2 + 5x + 6 = 0 \), we can factor it.

First, we look for two numbers that multiply to 6 (the constant term) and add up to 5 (the coefficient of the x term). These numbers are 2 and 3.

So, we can factor the equation as:

\( (x + 2)(x + 3) = 0 \)

Setting each factor equal to zero gives us the solutions:

\( x + 2 = 0 \) or \( x + 3 = 0 \)

Solving for x in each equation gives:

\( x = -2 \) or \( x = -3 \)

Therefore, the solutions to the equation \( x^2 + 5x + 6 = 0 \) are \( x = -2 \) and \( x = -3 \).<|im_end|>


In [20]:
model.save_pretrained("qwen3_lora_model")
tokenizer.save_pretrained("qwen3_lora_model")

('qwen3_lora_model/tokenizer_config.json',
 'qwen3_lora_model/special_tokens_map.json',
 'qwen3_lora_model/chat_template.jinja',
 'qwen3_lora_model/vocab.json',
 'qwen3_lora_model/merges.txt',
 'qwen3_lora_model/added_tokens.json',
 'qwen3_lora_model/tokenizer.json')

In [21]:
from unsloth import FastLanguageModel
finetuned_model, finetuned_tokenizer = FastLanguageModel.from_pretrained(
        model_name = "qwen3_lora_model",
        max_seq_length = 2048,
        load_in_4bit = True,
    )

==((====))==  Unsloth 2025.10.7: Fast Qwen3 patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [22]:
messages = [{"role" : "user", "content" : "Solve (x^2 + 5x + 6 = 0)."}]

text = finetuned_tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,
    enable_thinking = False,
)

from transformers import TextStreamer
_ = finetuned_model.generate(
    **finetuned_tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 2048,
    temperature = 0.7, top_p = 0.8, top_k = 20,
    streamer = TextStreamer(finetuned_tokenizer, skip_prompt = True),
)

To solve the quadratic equation \( x^2 + 5x + 6 = 0 \), we can factor it. First, we look for two numbers that multiply to 6 and add to 5. These numbers are 2 and 3. So, we can factor the equation as:

\( (x + 2)(x + 3) = 0 \)

Setting each factor equal to zero gives us the solutions:

\( x + 2 = 0 \) => \( x = -2 \)

\( x + 3 = 0 \) => \( x = -3 \)

So, the solutions to the equation are \( x = -2 \) and \( x = -3 \).<|im_end|>


In [23]:
messages = [
    {"role" : "user", "content" : "Solve (x^2 + 5x + 6 = 0)."}
]
text = finetuned_tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,
    enable_thinking = True,
)

from transformers import TextStreamer
_ = finetuned_model.generate(
    **finetuned_tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 2048,
    temperature = 0.7, top_p = 0.8, top_k = 20,
    streamer = TextStreamer(finetuned_tokenizer, skip_prompt = True),
)

<think>
Okay, so I need to solve the quadratic equation x² + 5x + 6 = 0. Hmm, let's see. I remember that quadratic equations can often be solved by factoring, completing the square, or using the quadratic formula. Since the equation is already in standard form (ax² + bx + c = 0), maybe factoring is the easiest way here. Let me try that first.

Factoring a quadratic usually involves finding two numbers that multiply to c (which is 6 here) and add up to b (which is 5). Let's list the pairs of numbers that multiply to 6. The factors of 6 are 1 and 6, 2 and 3. Let's check which pair adds up to 5. 2 and 3: 2 + 3 = 5. Perfect! So the equation can be factored as (x + 2)(x + 3) = 0. Wait, let me verify that. Expanding (x + 2)(x + 3) gives x² + 3x + 2x + 6 = x² + 5x + 6. Yep, that matches the original equation. Great.

Now, according to the zero product property, if the product of two factors is zero, then at least one of the factors must be zero. So, we set each factor equal to zero:

1. x + 2

In [24]:
%%writefile main.py
print("Hello, World!")

Writing main.py


In [27]:
import shutil

# Replace 'your_model_folder' with the path to your folder
folder_path = '/content/qwen3_lora_model'
shutil.make_archive(folder_path, 'zip', folder_path)

'/content/qwen3_lora_model.zip'

In [28]:
from google.colab import files

# This will download the zipped folder to your local machine
files.download(folder_path + '.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 137.1 MB/s eta 0:00:00


In [32]:
# !python main.py
!streamlit run main.py





  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.16.193.66:8501

  Stopping...
  Stopping...


In [35]:
!pip install nbconvert

In [38]:
!jupyter nbconvert --execute --to html "/content/Untitled1 (1).ipynb"

[NbConvertApp] Converting notebook /content/Untitled1 (1).ipynb to html
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.01s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
[NbConvertApp] ERROR | unhandled iopub msg: colab_request
[NbConvertApp] ERROR | unhandled iopub msg: colab_request
[NbConvertApp] ERROR | unhandled iopub msg: colab_request
[NbConvertApp] ERROR | unhandled iopub msg: colab_request
[NbConvertApp] ERROR | unhandled iopub msg: colab_request
[NbConvertApp] ERROR | unh